# 第 8 周练习：多代理交易搜寻系统

## 练习目标

用**模拟代理（Mock Agents）**独立演示第 8 周「价格正确（The Price is Right）」多代理架构：无需 Modal、HuggingFace 或外部 API，本地就能跑通一整轮「扫描 → 估价 → 排名 → 告警」。

## 架构（四个角色）

| 代理 | 职责 |
|------|------|
| **ScannerAgent** | 扫描交易（这里模拟 RSS 源，返回固定样例 Deal） |
| **EnsembleAgent** | 估计公平市场价格（这里用关键词规则模拟 LLM/RAG 合奏） |
| **MessagingAgent** | 对足够划算的机会发优惠提醒（写日志） |
| **PlanningAgent** | 编排整轮流程：扫描 → 估价 → 按折扣排序 → 阈值告警 |

## 流程

1. 规划代理启动一轮 `plan()`
2. 扫描仪返回若干 `Deal`
3. 合奏代理为每个 Deal 估一个 `estimate`，算 `discount = estimate - price`
4. 按折扣降序取最优；若 `discount >= DEAL_THRESHOLD` 则告警
5. 框架把结果写入本地 memory JSON，Gradio 表格展示历史机会

## 怎么跑

1. 先跑安装与导入单元格
2. 再跑数据模型、模拟代理、框架
3. 最后 `build_ui().launch()` 打开界面，点 **Scan for Deals**


In [ ]:
# ========== 安装依赖：Gradio 做界面，Pydantic 做结构化数据模型 ==========
# -q：安静安装，少刷屏
!pip install -q gradio pydantic


In [ ]:
# ========== 导入：标准库 + Gradio + Pydantic ==========

# os：读文件路径、判断 memory 文件是否存在
import os
# json：把 Opportunity 列表读写到本地 memory 文件
import json
# logging：代理用统一日志格式打印进度
import logging
# time：模拟扫描/估价耗时（sleep）
import time
# List / Optional：类型标注，方便读代码与 IDE 提示
from typing import List, Optional

# Gradio：快速搭 Web UI（Blocks / Dataframe / Button）
import gradio as gr
# BaseModel：Pydantic 基类，用来定义 Deal / Opportunity 等结构
from pydantic import BaseModel


## 数据模型

用 **Pydantic `BaseModel`** 固定三条业务对象的字段形状：

- `Deal`：一条交易（描述、标价、链接）
- `DealSelection`：扫描结果（多条 Deal）
- `Opportunity`：交易 + 估价 + 折扣，供排序与告警


In [ ]:
# ========== Pydantic 模型：给代理之间传递的数据定「形状」 ==========

# Deal：一条商品交易的最小字段集
class Deal(BaseModel):
    # 商品文字描述（后面合奏代理会按关键词估公允价）
    product_description: str
    # 当前标价（成交/广告价）
    price: float
    # 交易链接（示例用 example.com）
    url: str


# DealSelection：扫描代理一次返回的交易列表
class DealSelection(BaseModel):
    deals: List[Deal]


# Opportunity：在 Deal 上叠加估价与折扣，形成「可比较的机会」
class Opportunity(BaseModel):
    # 原始交易
    deal: Deal
    # 估计的公平市价（fair market estimate）
    estimate: float
    # 折扣幅度：estimate - deal.price（越大越划算）
    discount: float


## 模拟代理

下面用 **Mock*** 类代替真实 RSS / LLM / 推送服务：逻辑与第 8 周正式版同构，但输入输出是本地假数据，方便离线演示编排。


In [ ]:
# ========== 模拟代理：Scanner / Ensemble / Messaging / Planning ==========

# 所有 Mock 代理的基类：统一 name + log
class MockAgent:
    # 默认名字；子类会覆盖
    name = "MockAgent"

    def log(self, message: str):
        # 日志前缀带代理名，方便在 Gradio/终端里区分谁在说话
        logging.info(f"[{self.name}] {message}")


# 扫描代理：假装读 RSS，返回三条固定样例交易
class MockScannerAgent(MockAgent):
    name = "ScannerAgent"

    def scan(self, memory=None) -> DealSelection:
        # memory 参数保留与真实版签名一致；Mock 里未使用
        self.log("Scanning RSS feeds for deals...")
        # 模拟网络扫描耗时
        time.sleep(0.5)
        # 硬编码三条 Deal（描述/标价/URL 保持英文原样，便于对照真实数据形态）
        deals = [
            Deal(
                product_description="Apple iPad Pro 11-inch 256GB WiFi - M2 chip, Liquid Retina display, Face ID.",
                price=749.99,
                url="https://example.com/ipad",
            ),
            Deal(
                product_description="Sony WH-1000XM5 Wireless Noise Cancelling Headphones - 30hr battery.",
                price=329.99,
                url="https://example.com/sony",
            ),
            Deal(
                product_description="Logitech MX Master 3S Wireless Mouse - Ergonomic, 8K DPI.",
                price=69.99,
                url="https://example.com/logitech",
            ),
        ]
        self.log(f"Found {len(deals)} deals")
        # 包进 DealSelection 再返回
        return DealSelection(deals=deals)


# 合奏/估价代理：用关键词规则假装「LLM + RAG」估公允价
class MockEnsembleAgent(MockAgent):
    name = "EnsembleAgent"

    def price(self, description: str) -> float:
        self.log("Estimating fair market price...")
        # 模拟模型推理耗时
        time.sleep(0.3)
        # 按描述里的品牌/型号关键词返回预设公允价
        if "iPad" in description:
            return 899.0
        elif "Sony" in description:
            return 398.0
        elif "Logitech" in description:
            return 99.0
        # 未命中关键词时的兜底估价
        return 150.0


# 消息代理：对选中的机会打一条告警日志（真实版可能是 Pushover/SMS）
class MockMessagingAgent(MockAgent):
    name = "MessagingAgent"

    def alert(self, opportunity: Opportunity):
        # 只展示折扣与描述前 40 字符，避免日志过长
        self.log(f"Alert: ${opportunity.discount:.2f} discount on {opportunity.deal.product_description[:40]}...")


# 规划代理：编排一轮完整周期
class MockPlanningAgent(MockAgent):
    name = "PlanningAgent"
    # 折扣阈值：只有最优机会的 discount >= 50 才发告警
    DEAL_THRESHOLD = 50

    def __init__(self):
        # 组合三个子代理（依赖注入的简化写法：直接 new）
        self.scanner = MockScannerAgent()
        self.ensemble = MockEnsembleAgent()
        self.messenger = MockMessagingAgent()

    def plan(self, memory=None) -> Optional[Opportunity]:
        # memory 可为历史机会列表；Mock 扫描未用，但签名对齐真实 PlanningAgent
        if memory is None:
            memory = []
        self.log("Starting planning cycle")
        # 1) 扫描
        selection = self.scanner.scan(memory)
        # 没有交易则本轮结束
        if not selection or not selection.deals:
            return None
        opportunities = []
        # 2) 逐条估价并算折扣
        for deal in selection.deals:
            estimate = self.ensemble.price(deal.product_description)
            discount = estimate - deal.price
            opportunities.append(Opportunity(deal=deal, estimate=estimate, discount=discount))
        # 3) 按折扣从高到低排序
        opportunities.sort(key=lambda x: x.discount, reverse=True)
        best = opportunities[0]
        self.log(f"Best deal: ${best.discount:.2f} discount")
        # 4) 超过阈值才告警
        if best.discount >= self.DEAL_THRESHOLD:
            self.messenger.alert(best)
        return best


## 交易代理框架

`MockDealAgentFramework` 负责：**初始化规划代理**、**跑一轮 plan**、以及把结果 **持久化到 JSON memory**（文件名 `week8_mock_memory.json`）。


In [ ]:
# ========== 框架：memory 读写 + 调用 PlanningAgent ==========

class MockDealAgentFramework:
    # 本地记忆文件：保存历史 Opportunity（Pydantic model_dump）
    MEMORY_FILE = "week8_mock_memory.json"

    def __init__(self):
        # 启动时先从磁盘恢复历史机会
        self.memory: List[Opportunity] = self._read_memory()
        # 规划代理懒加载（第一次 run 时再创建）
        self.planner: Optional[MockPlanningAgent] = None

    def _read_memory(self) -> List[Opportunity]:
        # 文件存在则尝试 JSON → Opportunity 列表
        if os.path.exists(self.MEMORY_FILE):
            try:
                with open(self.MEMORY_FILE, "r") as f:
                    data = json.load(f)
                # **item 解包字段构造 Pydantic 模型
                return [Opportunity(**item) for item in data]
            except Exception:
                # 损坏/格式不对时退回空列表，避免整框架起不来
                return []
        return []

    def _write_memory(self):
        # model_dump：Pydantic v2 序列化为可 JSON 的 dict
        data = [opp.model_dump() for opp in self.memory]
        with open(self.MEMORY_FILE, "w") as f:
            json.dump(data, f, indent=2)

    def init_agents(self):
        # 只初始化一次 planner
        if not self.planner:
            self.planner = MockPlanningAgent()
            logging.info("Agent framework initialized")

    def run(self) -> List[Opportunity]:
        # 确保代理就绪 → 跑一轮 plan → 有结果则追加并落盘
        self.init_agents()
        result = self.planner.plan(memory=self.memory)
        if result:
            self.memory.append(result)
            self._write_memory()
        # 始终返回完整 memory，方便 UI 刷新表格
        return self.memory


## Gradio 用户界面

用 **Gradio Blocks** 搭一个最小控制台：表格展示历史机会，按钮触发一轮扫描。界面上的英文文案是 UI 字符串（保留原文，避免改行为/截图对照）。


In [ ]:
# ========== Gradio UI：表格 +「Scan for Deals」按钮 ==========

# 配置日志：INFO 级别，带时分秒，方便在界面侧看到代理 log
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s", datefmt="%H:%M:%S")

# 全局框架实例：多次点击共用同一份 memory
framework = MockDealAgentFramework()


def opps_to_table(opps: List[Opportunity]) -> List[List]:
    # 把 Opportunity 列表转成 Gradio Dataframe 需要的二维列表
    return [
        [o.deal.product_description, f"${o.deal.price:.2f}", f"${o.estimate:.2f}", f"${o.discount:.2f}", o.deal.url]
        for o in opps
    ]


def scan_for_deals():
    # 按钮回调：跑一轮框架，返回刷新后的表格数据
    opps = framework.run()
    return opps_to_table(opps)


def build_ui():
    # Blocks：声明式搭页面；title 会出现在浏览器标签
    with gr.Blocks(title="The Price is Right - Mock") as ui:
        # UI 文案保持英文原样（可运行界面字符串，不翻译）
        gr.Markdown("## The Price is Right - Multi-Agent Deal Hunting (Mock)")
        gr.Markdown("Click **Scan for Deals** to run one planning cycle. The framework scans mock deals, estimates prices, and surfaces the best opportunity.")
        # Dataframe：五列展示描述/标价/估价/折扣/链接
        table = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            col_count=5,
            row_count=10,
            # 初始值：已从 memory 文件恢复的历史机会
            value=opps_to_table(framework.memory),
        )
        btn = gr.Button("Scan for Deals", variant="primary")
        # 点击 → 调用 scan_for_deals → 输出写回 table
        btn.click(scan_for_deals, outputs=table)
    return ui


In [ ]:
# ========== 启动 Gradio：浏览器里点按钮跑一轮规划 ==========
build_ui().launch()
